# Features not yet available in the Interactions API

This notebook documents Gemini API features that are currently only available via `client.models.generate_content` (the `generateContent` API) and are **not yet supported** in the Interactions API (`client.interactions.create`).

Use this as a reference when deciding which API to use for a specific feature.

## Table of Contents

1. [Automatic Function Calling](#auto_fc)
2. [Function Calling Modes (AUTO/ANY/NONE)](#fc_modes)
3. [Grounding Sources for Maps](#maps_sources)
4. [Safety Settings Configuration](#safety)
5. [Explicit Context Caching](#caching)
6. [Batch API](#batch)

<a name="auto_fc"></a>
## Automatic Function Calling

**Status:** Only available with `client.chats.create` (which uses `generateContent`)

The Python SDK's `ChatSession` can automatically execute function calls from the model:

1. Model returns a `FunctionCall`
2. SDK finds and executes the corresponding Python function
3. SDK sends the `FunctionResponse` back to the model
4. Returns the final text response

**With the Interactions API**, you must handle function calls manually:
- Check `interaction.steps` for `function_call` type entries
- Execute functions yourself
- Send results back via `previous_interaction_id` with `function_result` input

### generateContent example (automatic):

In [ ]:
# Automatic function calling with client.chats (generateContent)
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

def multiply(a: float, b: float):
    """returns a * b."""
    print(f"  >> multiply({a}, {b})")
    return a * b

chat = client.chats.create(
    model="gemini-3-flash-preview",
    config={
        "tools": [multiply],
        "automatic_function_calling": {"disable": False},  # Enabled by default
    },
)

response = chat.send_message("What is 7 times 6?")
print(response.text)  # "42" — function was called automatically

### Interactions API equivalent (manual):

In [ ]:
# Manual function calling with interactions API
interaction = client.interactions.create(
    model="gemini-3-flash-preview",
    input="What is 7 times 6?",
    tools=[types.FunctionDeclaration.from_callable(callable=multiply, client=client).to_json_dict()],
)

# Check for function calls and handle manually
for step in interaction.steps:
    if step.type == "function_call":
        print(f"Model wants to call: {step.name}({step.arguments})")
        result = multiply(**step.arguments)

        # Send result back
        interaction_2 = client.interactions.create(
            model="gemini-3-flash-preview",
            previous_interaction_id=interaction.id,
            input={
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": str(result),
            },
        )
        print(interaction_2.steps[-1].content[0].text)

<a name="fc_modes"></a>
## Function Calling Modes (AUTO/ANY/NONE)

**Status:** Only available with `generateContent` via `tool_config`

The `tool_config` parameter lets you control when and which functions the model can call:

| Mode | Behavior | Interactions API workaround |
|------|----------|-----------------------------|
| **AUTO** | Model decides whether to call functions or respond with text | Default behavior — just pass `tools` |
| **NONE** | Model cannot call any functions | Don't pass `tools` |
| **ANY** | Model **must** call at least one function | No direct equivalent; use prompting |

### generateContent example with tool_config:

In [ ]:
from google.genai import types

# ANY mode — force the model to call a function
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="What is 7 times 6?",
    config=types.GenerateContentConfig(
        tools=[multiply],
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode="ANY",
                # Optional: restrict to specific functions
                # allowed_function_names=["multiply"],
            ),
        ),
    ),
)

# The response will contain a FunctionCall part
for part in response.candidates[0].content.parts:
    if part.function_call:
        print(f"Function call: {part.function_call.name}({part.function_call.args})")

### `allowed_function_names`

With ANY mode, you can restrict which functions the model is allowed to call:

```python
tool_config=types.ToolConfig(
    function_calling_config=types.FunctionCallingConfig(
        mode="ANY",
        allowed_function_names=["find_movies"],  # Only this function
    ),
)
```

This is useful when the application state dictates that the next step must involve a specific action.

<a name="safety"></a>
## Safety Settings Configuration

**Status:** Not available in the Interactions API

With `generateContent`, you can configure safety thresholds per harm category:

```python
from google.genai import types

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="...",
    config=types.GenerateContentConfig(
        safety_settings=[
            types.SafetySetting(
                category="HARM_CATEGORY_HARASSMENT",
                threshold="BLOCK_ONLY_HIGH",
            ),
        ],
    ),
)
```

With the Interactions API, safety is handled by the model internally and cannot be configured per-request.

<a name="caching"></a>
## Explicit Context Caching

**Status:** Not available in the Interactions API (implicit caching is automatic)

With `generateContent`, you can explicitly create and manage caches:

```python
# Create a cache
cache = client.caches.create(
    model="gemini-3-flash-preview",
    contents=[large_document],
    config=types.CreateCachedContentConfig(
        display_name="my-cache",
        ttl="3600s",
    ),
)

# Use the cache
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Summarize this document.",
    config=types.GenerateContentConfig(
        cached_content=cache.name,
    ),
)
```

With the Interactions API, caching is handled **implicitly** via `previous_interaction_id` — the server automatically reuses context from previous turns.

<a name="maps_sources"></a>
## Grounding Sources for Maps

**Status:** Not available in the Interactions API

With `generateContent`, grounding responses include `grounding_metadata` with source citations and a Google Maps widget:

```python
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Find me a good coffee shop nearby",
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_maps=types.GoogleMaps())],
    ),
)

# Access grounding metadata
grounding = response.candidates[0].grounding_metadata

# Sources
for chunk in grounding.grounding_chunks:
    print(chunk.web.title, chunk.web.uri)

# Render the Maps widget (for web apps)
html_widget = grounding.search_entry_point.rendered_content
```

With the Interactions API, the grounding sources and Maps widget data are not yet available in the response.

<a name="batch"></a>
## Batch API

**Status:** Only available with `generateContent`

The Batch API lets you process large numbers of requests asynchronously:

```python
batch_job = client.batches.create(
    model="gemini-3-flash-preview",
    src="gs://bucket/input.jsonl",
    config=types.CreateBatchJobConfig(
        dest="gs://bucket/output.jsonl",
    ),
)
```

There is no batch equivalent for the Interactions API. Use `client.batches` with `generateContent` format for bulk processing.